# Appariage stable — qui peut tricher dans Gale–Shapley ?

**GameTheory-16i** (*matching*) approfondit le [carnet 16](GameTheory-16-MechanismDesign-Python.ipynb), qui introduit l’algorithme, la stabilité et l’avantage du côté proposant.

Gale et Shapley (1962) ont posé l’algorithme d’acceptation différée. Ce carnet exécute la procédure, vérifie indépendamment la stabilité et mesure les déclarations sur deux instances 3×3. Il distingue le théorème d’optimalité du côté proposant des résultats d’énumération propres aux exemples. Gibbard–Satterthwaite porte sur le choix social et n’est pas directement le théorème de manipulation de ce modèle.

**Objectifs** : exécuter Gale–Shapley, vérifier les paires bloquantes, comparer les résultats selon le côté proposant et examiner des déclarations stratégiques.

**Prérequis** : listes de préférences strictes complètes, dictionnaires et permutations.

**Durée estimée** : 45 minutes.

**Navigation** : [Version Python](GameTheory-16i-Appariage-Stable-Gale-Shapley-Python.ipynb) | [Index GameTheory](README.md) | Précédent : [GameTheory-16h — Coopération LLM](GameTheory-16h-Cooperation-LLM-Python.ipynb) | Base : [carnet 16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb).


In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Linq;
using System.Globalization;
CultureInfo.CurrentCulture=CultureInfo.InvariantCulture;
record PairingProblem(List<string> Proposers, Dictionary<string,List<string>> ProposerPrefs, List<string> Respondents, Dictionary<string,List<string>> RespondentPrefs);
record BlockingPair(string Proposer,string Respondent);
record ManipulationResult(List<string>? Declaration,Dictionary<string,string>? Match);
void ValidateProblem(IReadOnlyList<string> proposers,IReadOnlyDictionary<string,List<string>> pp,IReadOnlyList<string> respondents,IReadOnlyDictionary<string,List<string>> rp){
 if(proposers.Count!=respondents.Count||proposers.Distinct().Count()!=proposers.Count||respondents.Distinct().Count()!=respondents.Count||proposers.Intersect(respondents).Any())throw new ArgumentException("Les côtés doivent être disjoints et de même taille.");
 if(!pp.Keys.OrderBy(x=>x).SequenceEqual(proposers.OrderBy(x=>x))||!rp.Keys.OrderBy(x=>x).SequenceEqual(respondents.OrderBy(x=>x)))throw new ArgumentException("Clés de préférences incomplètes ou inconnues.");
 foreach(var p in proposers)if(pp[p].Count!=respondents.Count||pp[p].Distinct().Count()!=respondents.Count||!pp[p].OrderBy(x=>x).SequenceEqual(respondents.OrderBy(x=>x)))throw new ArgumentException("Chaque liste doit être une permutation complète.");
 foreach(var r in respondents)if(rp[r].Count!=proposers.Count||rp[r].Distinct().Count()!=proposers.Count||!rp[r].OrderBy(x=>x).SequenceEqual(proposers.OrderBy(x=>x)))throw new ArgumentException("Chaque liste doit être une permutation complète.");
}
Dictionary<string,string> GaleShapley(IReadOnlyList<string> proposers,Dictionary<string,List<string>> pp,IReadOnlyList<string> respondents,Dictionary<string,List<string>> rp){
 ValidateProblem(proposers,pp,respondents,rp);var rank=rp.ToDictionary(kv=>kv.Key,kv=>kv.Value.Select((p,i)=>(p,i)).ToDictionary(x=>x.p,x=>x.i));
 var free=proposers.ToList();var next=proposers.ToDictionary(x=>x,_=>0);var current=new Dictionary<string,string>();var match=new Dictionary<string,string>();
 while(free.Count>0){string p=free[^1];free.RemoveAt(free.Count-1);while(next[p]<pp[p].Count){string r=pp[p][next[p]++];if(!current.TryGetValue(r,out var q)){current[r]=p;match[p]=r;break;}if(rank[r][p]<rank[r][q]){current[r]=p;match[p]=r;match.Remove(q);free.Add(q);break;}}}
 return match;
}
BlockingPair? FindBlockingPair(IReadOnlyDictionary<string,string> match,Dictionary<string,List<string>> pp,Dictionary<string,List<string>> rp){
 if(match.Count==0)return null;var reverse=match.ToDictionary(kv=>kv.Value,kv=>kv.Key);foreach(var (p,r) in match){foreach(var better in pp[p]){if(better==r)break;if(!reverse.TryGetValue(better,out var incumbent)||rp[better].IndexOf(p)<rp[better].IndexOf(incumbent))return new BlockingPair(p,better);}}return null;
}
bool IsStable(IReadOnlyList<string> ps,Dictionary<string,List<string>> pp,IReadOnlyList<string> rs,Dictionary<string,List<string>> rp,IReadOnlyDictionary<string,string> match){
 ValidateProblem(ps,pp,rs,rp);if(match.Count!=ps.Count||!match.Keys.OrderBy(x=>x).SequenceEqual(ps.OrderBy(x=>x))||match.Values.Distinct().Count()!=rs.Count||!match.Values.OrderBy(x=>x).SequenceEqual(rs.OrderBy(x=>x)))return false;return FindBlockingPair(match,pp,rp) is null;
}
List<Dictionary<string,string>> AllBijections(List<string> ps,List<string> rs){var result=new List<Dictionary<string,string>>();void Rec(int i,List<string> remaining,Dictionary<string,string> current){if(i==ps.Count){result.Add(new(current));return;}foreach(var r in remaining){current[ps[i]]=r;Rec(i+1,remaining.Where(x=>x!=r).ToList(),current);current.Remove(ps[i]);}}Rec(0,rs.ToList(),new());return result;}
List<List<string>> Permute(List<string> xs){var result=new List<List<string>>();void Rec(List<string> left,List<string> made){if(left.Count==0){result.Add(made.ToList());return;}foreach(var x in left.ToList()){left.Remove(x);made.Add(x);Rec(left,made);made.RemoveAt(made.Count-1);left.Add(x);}}Rec(xs.ToList(),new());return result;}
string Show(IReadOnlyDictionary<string,string> m)=>"{"+string.Join(", ",m.OrderBy(kv=>kv.Key).Select(kv=>$"{kv.Key}: {kv.Value}"))+"}";
Console.WriteLine($"BCL .NET {Environment.Version} ; Gale–Shapley, oracle de stabilité et énumérateur de bijections prêts.");

BCL .NET 10.0.14 ; Gale–Shapley, oracle de stabilité et énumérateur de bijections prêts.


## 1. Le problème de l'appariage stable

On a **n hommes** et **n femmes**, chacun avec une **liste de préférences** complète et strictement ordonnée sur l'autre sexe. Un **appariage** associe chaque homme à une femme (bijection).

Une **paire bloquante** est une paire (homme, femme) *non* appariés ensemble, où **les deux** préféreraient s'associer l'un à l'autre plutôt qu'à leur partenaire actuel. Un appariage est **stable** s'il n'admet **aucune** paire bloquante.

L'intuition : un appariage instable « s'effondre » — la paire bloquante s'unit et rompt les deux mariages existants. La question est : *existe-t-il toujours un appariage stable, et comment le trouver ?*

In [2]:
var HOMMES=new List<string>{"m1","m2","m3"};var FEMMES=new List<string>{"w1","w2","w3"};
var H_prefs=new Dictionary<string,List<string>>{{"m1",new(){"w1","w2","w3"}},{"m2",new(){"w2","w3","w1"}},{"m3",new(){"w3","w1","w2"}}};
var F_prefs=new Dictionary<string,List<string>>{{"w1",new(){"m3","m1","m2"}},{"w2",new(){"m1","m2","m3"}},{"w3",new(){"m2","m3","m1"}}};
ValidateProblem(HOMMES,H_prefs,FEMMES,F_prefs);foreach(var h in HOMMES)Console.WriteLine($"{h} : {string.Join(" > ",H_prefs[h])}");Console.WriteLine();foreach(var f in FEMMES)Console.WriteLine($"{f} : {string.Join(" > ",F_prefs[f])}");

m1 : w1 > w2 > w3


m2 : w2 > w3 > w1


m3 : w3 > w1 > w2


w1 : m3 > m1 > m2


w2 : m1 > m2 > m3


w3 : m2 > m3 > m1


### Lecture de l'instance A

La sortie ci-dessus expose la structure du conflit : chaque homme a un premier choix **distinct** (`m1→w1`, `m2→w2`, `m3→w3`) — pas de compétition apparente — et chaque femme aussi (`w1→m3`, `w2→m1`, `w3→m2`). Mais ces deux « mondes idéaux » ne coïncident pas : `w1` veut `m3`, qui préfère `w3` ; `w2` veut `m1`, qui préfère `w1`. Personne n'est partagé, et pourtant aucun accord évident n'émerge — c'est exactement la situation où le critère réaliste n'est plus l'unanimité mais la **stabilité** : un appariement qu'aucun couple ne veut quitter ensemble.

Concrètement, cette instance admet **deux** appariements stables, et c'est ce qui rend l'exemple instructif : le premier, `{m1: w1, m2: w2, m3: w3}`, et le second, `{m1: w2, m2: w3, m3: w1}`. Vérifions pourquoi chacun est stable : on cherche une **paire bloquante**, c'est-à-dire un homme et une femme qui s'aiment mutuellement plus que leur partenaire actuel. Dans le premier, `w1` aimerait `m3`, mais `m3` est engagé à `w3` qu'il préfère — le couple `(m3, w1)` n'est donc pas bloquant, car un des deux (en l'occurrence `m3`) n'a aucune envie de fuir ; le même raisonnement élimine chaque autre frustration. Le raisonnement tient symétriquement dans le second appariement. Deux structures stables coexistent et aucune n'est « la » solution : le choix entre elles dépendra de **qui propose**, comme le mesurera la section suivante. C'est ce qui distingue l'appariage stable d'un problème d'optimisation à solution unique.


## 2. L'algorithme de Gale–Shapley (*deferred acceptance*)

L'algorithme fait **proposer** à un côté (ici les hommes) :

1. un homme **libre** propose à la **première** femme de sa liste qu'il n'a pas encore sollicitée ;
2. si la femme est **libre**, elle accepte temporairement ;
3. si elle est **engagée**, elle garde le **préféré** entre son prétendant actuel et le nouveau, et libère l'autre ;
4. on répète jusqu'à ce que plus personne ne soit libre.

Chaque engagement est **réversible** (d'où *deferred acceptance*) : une femme peut « lâcher » son partenaire actuel pour un meilleur.

Pour ancrer le mécanisme, suivons les premières itérations sur l'instance A. Au départ, les trois hommes sont libres et proposent chacun à son **premier** choix : `m1→w1`, `m2→w2`, `m3→w3`. Les trois femmes sont libres, elles acceptent chacune, et plus d'homme n'est libre — l'algorithme **s'arrête immédiatement**, avec l'appariement `{m1: w1, m2: w2, m3: w3}`. Sur cette instance il n'y a donc aucune itération de « lâcher » : les premiers choix des hommes sont distincts ; les premiers choix des femmes le sont également. C'est une lecture utile du résultat 1, mais ne la sur-généralisons pas : d'autres instances (la C, plus bas) font **tourner** la boucle, avec des engagements réversibles en chaîne, jusqu'à l'impasse. C'est le *deferred acceptance* qui rend ce parcours toujours bien défini : un engagement tenu pour l'instant, révocable dès qu'un meilleur prétendant arrive.

L'algorithme **termine toujours** (chaque proposition élimine définitivement une paire homme-femme, et il en est un nombre fini) et produit un appariage **stable**.


### Exercice 1 — Reconstruire l’acceptation différée

Complétez le squelette de Gale–Shapley. Vérifiez l’instance A, un singleton et le cas vide ; une réponse vide ne convient que pour `n=0`.

In [3]:
#nullable enable
// Exercice 1 — stub et vérification de deux fixtures (vide et singleton inclus).
Dictionary<string,string>? GaleShapleyEtudiant(IReadOnlyList<string> ps,Dictionary<string,List<string>> pp,IReadOnlyList<string> rs,Dictionary<string,List<string>> rp)=>null; // TODO étudiant : boucle deferred acceptance.
var gsAStudent=GaleShapleyEtudiant(HOMMES,H_prefs,FEMMES,F_prefs);
var oneStudent=GaleShapleyEtudiant(new List<string>{"p"},new Dictionary<string,List<string>>{{"p",new List<string>{"r"}}},new List<string>{"r"},new Dictionary<string,List<string>>{{"r",new List<string>{"p"}}});
var emptyStudent=GaleShapleyEtudiant(new List<string>(),new Dictionary<string,List<string>>(),new List<string>(),new Dictionary<string,List<string>>());
bool gsStudentPresent=gsAStudent is not null||oneStudent is not null||emptyStudent is not null;
bool gsExercisePass=gsAStudent is not null&&IsStable(HOMMES,H_prefs,FEMMES,F_prefs,gsAStudent)&&gsAStudent.SequenceEqual(GaleShapley(HOMMES,H_prefs,FEMMES,F_prefs))&&oneStudent!=null&&oneStudent.Count==1&&oneStudent["p"]=="r"&&emptyStudent!=null&&emptyStudent.Count==0;
Console.WriteLine($"Exercice 1 — {(gsStudentPresent?(gsExercisePass?"réponse validée":"réponse incorrecte"):"à compléter")}; réponse contrôlée sur A, singleton et n=0={gsExercisePass}; côté vide valide uniquement pour n=0.");

Exercice 1 — à compléter; réponse contrôlée sur A, singleton et n=0=False; côté vide valide uniquement pour n=0.


In [4]:
var appariage=GaleShapley(HOMMES,H_prefs,FEMMES,F_prefs);var pb=FindBlockingPair(appariage,H_prefs,F_prefs);
Console.WriteLine($"Appariage produit (hommes proposent) : {Show(appariage)}");Console.WriteLine($"Paire bloquante : {(pb is null?"aucune":"("+pb.Proposer+", "+pb.Respondent+")")} → {(IsStable(HOMMES,H_prefs,FEMMES,F_prefs,appariage)?"stable":"INSTABLE")}");
var duplicatePrefs=new Dictionary<string,List<string>>(H_prefs);duplicatePrefs["m1"]=new(){"w1","w1","w3"};bool badInputRejected=false;try{_ = GaleShapley(HOMMES,duplicatePrefs,FEMMES,F_prefs);}catch(ArgumentException){badInputRejected=true;}
bool unbalancedRejected=false;try{_ = GaleShapley(HOMMES,H_prefs,new List<string>{"w1","w2"},F_prefs);}catch(ArgumentException){unbalancedRejected=true;}
Console.WriteLine($"Validation des préférences : liste dupliquée rejetée={badInputRejected}; côtés déséquilibrés rejetés={unbalancedRejected}; l'instance A est stable={IsStable(HOMMES,H_prefs,FEMMES,F_prefs,appariage)}.");

Appariage produit (hommes proposent) : {m1: w1, m2: w2, m3: w3}


Paire bloquante : aucune → stable


Validation des préférences : liste dupliquée rejetée=True; côtés déséquilibrés rejetés=True; l'instance A est stable=True.


### Résultat 1 — Existence

La mesure le confirme : l'algorithme **termine** et l'appariement produit — ici `{m1: w1, m2: w2, m3: w3}` — ne contient **aucune paire bloquante** (`trouver_paire_bloquante` retourne `None`).

Un détail de la sortie mérite une lecture : chaque homme obtient son **premier choix**, et pourtant l'appariement n'est pas « idéal pour tous ». `w1` aurait préféré `m3` — mais `m3` a obtenu `w3`, *son* premier choix, donc le couple `(m3, w1)` n'est pas bloquant : un seul des deux a envie de fuir. **Stable ≠ unanime** : la stabilité n'exige l'absence que des paires *mutuellement* insatisfaites.

Pourquoi la termination est-elle si forte ici ? Chaque fois qu'un homme est rejeté, il ne **reproposera jamais** à cette femme : sa position dans sa propre liste ne fait que **descendre**, et une liste est finie. Le nombre total de propositions possibles est donc borné par `n²`, et aucune boucle ne peut tourner indéfiniment. La termination n'est pas une heuristique : c'est elle qui **prouve** l'existence, car à l'instant où l'algorithme s'arrête, il tient un appariement sans paire bloquante.

C'est le théorème d'**existence** de Gale–Shapley : *pour tout problème d'appariement stable, il existe au moins un appariement stable* — et l'algorithme le trouve en **temps polynomial** (O(n²) propositions au plus).


### Exercice 2 — Vérifier la stabilité

Implémentez un oracle qui renvoie une paire bloquante pour le fixture instable et `null` pour l’appariement stable. Les deux cas rendent un stub constant discriminable.

In [5]:
#nullable enable
// Exercice 2 — stub de vérification et fixtures stable / instable.
BlockingPair? TrouverPaireBloquanteEtudiant(IReadOnlyDictionary<string,string> match,Dictionary<string,List<string>> pp,Dictionary<string,List<string>> rp)=>null; // TODO étudiant
BlockingPair? stableAnswer=TrouverPaireBloquanteEtudiant(appariage,H_prefs,F_prefs);
var unstable=new Dictionary<string,string>{{"m1","w2"},{"m2","w1"},{"m3","w3"}};BlockingPair? unstableAnswer=TrouverPaireBloquanteEtudiant(unstable,H_prefs,F_prefs);
bool blockingPresent=stableAnswer is not null||unstableAnswer is not null;bool blockingPass=stableAnswer is null&&unstableAnswer is not null&&IsStable(HOMMES,H_prefs,FEMMES,F_prefs,appariage)&&!IsStable(HOMMES,H_prefs,FEMMES,F_prefs,unstable);
Console.WriteLine($"Exercice 2 — {(blockingPresent?(blockingPass?"réponse validée":"réponse incorrecte"):"à compléter")}; stable A renvoie null et le fixture instable renvoie un blocage={blockingPass}; null est légitime uniquement pour un matching stable.");

Exercice 2 — à compléter; stable A renvoie null et le fixture instable renvoie un blocage=False; null est légitime uniquement pour un matching stable.


## 3. Comparer les deux côtés proposant

Exécutez Gale–Shapley dans les deux sens, puis énumérez les six bijections pour compter les appariages stables de l’instance A.

In [6]:
var m_opt=GaleShapley(HOMMES,H_prefs,FEMMES,F_prefs);var fRaw=GaleShapley(FEMMES,F_prefs,HOMMES,H_prefs);var f_opt=fRaw.ToDictionary(kv=>kv.Value,kv=>kv.Key);
Console.WriteLine($"Équilibre hommes proposent : {Show(m_opt)}");Console.WriteLine($"Équilibre femmes proposent : {Show(f_opt)}");Console.WriteLine($"Identiques ? {m_opt.SequenceEqual(f_opt)}");Console.WriteLine("Rang du partenaire de chaque homme (1 = premier choix) :");foreach(var hName in HOMMES)Console.WriteLine($"  {hName} : {H_prefs[hName].IndexOf(m_opt[hName])+1} si hommes proposent, {H_prefs[hName].IndexOf(f_opt[hName])+1} si femmes proposent");
var stableA=AllBijections(HOMMES,FEMMES).Where(m=>IsStable(HOMMES,H_prefs,FEMMES,F_prefs,m)).ToList();Console.WriteLine($"Oracle indépendant : {stableA.Count} appariages stables sur {AllBijections(HOMMES,FEMMES).Count} bijections.");

Équilibre hommes proposent : {m1: w1, m2: w2, m3: w3}


Équilibre femmes proposent : {m1: w2, m2: w3, m3: w1}


Identiques ? False


Rang du partenaire de chaque homme (1 = premier choix) :


  m1 : 1 si hommes proposent, 2 si femmes proposent


  m2 : 1 si hommes proposent, 2 si femmes proposent


  m3 : 1 si hommes proposent, 2 si femmes proposent


Oracle indépendant : 2 appariages stables sur 6 bijections.


### Résultat 2 — L'avantage du côté proposeur

La sortie montre **deux appariements stables distincts** sur la même instance : `{m1: w1, m2: w2, m3: w3}` quand les hommes proposent, `{m1: w2, m2: w3, m3: w1}` quand les femmes proposent (`Identiques ? False`).

Le tableau de rangs imprimé est la lecture clé : **chaque** homme passe du 1er choix quand il propose au 2e quand il ne propose plus — aucun n'est indifférent au rôle. En croisant ce tableau avec les listes affichées en section 1, la contrepartie côté femmes est le **miroir exact** : dans l'équilibre « hommes proposent », `w1`, `w2` et `w3` sont toutes à leur **2e choix** ; dans l'équilibre « femmes proposent », elles sont toutes à leur **1er**. L'instance A réalise ainsi le cas d'école complet : le côté proposeur obtient son **meilleur** appariement stable, le côté répondant son **pire** (théorème d'**optimalité** de Gale–Shapley). *Dans la conception du mécanisme, donner le pouvoir de proposer avantage celui qui propose.*

Formulé en général, ce théorème (souvent attribué à Gale–Shapley–Roth) dit plus que cette instance ne montre : quand les hommes proposent, **chaque** homme obtient le **meilleur** partenaire qu'il peut espérer parmi **tous** les appariements stables, et **chaque** femme son **pire**. « Meilleur » et « pire » se comparent sur l'ensemble **complet** des équilibres stables, pas seulement sur les deux que nous voyons ici. Sur l'instance A il n'y en a que deux, si bien que « meilleur » et « pire » tombent exactement sur les deux extrêmes observés. Mais sur des instances plus grandes, le nombre d'équilibres stables peut exploser (il n'existe pas de borne polynomiale sur ce nombre), et l'écart entre le meilleur et le pire stable d'un même agent devient considérable. C'est ce qui donne tout son poids à la règle « qui propose » : l'avantage du proposeur n'est pas un artifact de symétrie de cet exemple, c'est une **structure** du mécanisme.


## 4. Instance C : déclarations stratégiques

Cette seconde instance fait parcourir la file de propositions et permet de comparer les résultats d’une recherche finie de mensonges aux théorèmes du modèle.

In [7]:
var C_H=new Dictionary<string,List<string>>{{"m1",new(){"w2","w1","w3"}},{"m2",new(){"w2","w3","w1"}},{"m3",new(){"w1","w2","w3"}}};var C_F=new Dictionary<string,List<string>>{{"w1",new(){"m2","m1","m3"}},{"w2",new(){"m3","m1","m2"}},{"w3",new(){"m1","m2","m3"}}};
var C_honnete=GaleShapley(C_H.Keys.ToList(),C_H,C_F.Keys.ToList(),C_F);Console.WriteLine($"Appariage honnête (hommes proposent) : {Show(C_honnete)}");
int RangM(string h,string f)=>C_H[h].IndexOf(f)+1;int RangF(string f,string h)=>C_F[f].IndexOf(h)+1;
int menGain=0;foreach(var h in C_H.Keys){int baseline=RangM(h,C_honnete[h]);foreach(var perm in Permute(C_H[h])){var altered=new Dictionary<string,List<string>>(C_H);altered[h]=perm;var match=GaleShapley(C_H.Keys.ToList(),altered,C_F.Keys.ToList(),C_F);if(RangM(h,match[h])<baseline){menGain++;break;}}}
var reversedOrderMatch=GaleShapley(C_H.Keys.Reverse().ToList(),C_H,C_F.Keys.ToList(),C_F);bool orderIndependent=C_honnete.Count==reversedOrderMatch.Count&&C_honnete.All(kv=>reversedOrderMatch.TryGetValue(kv.Key,out var partner)&&partner==kv.Value);
int stableCountC=AllBijections(C_H.Keys.ToList(),C_F.Keys.ToList()).Count(m=>IsStable(C_H.Keys.ToList(),C_H,C_F.Keys.ToList(),C_F,m));Console.WriteLine($"Hommes pouvant gagner par déclaration : {menGain} sur {C_H.Count} ; C porte {stableCountC} bijections stables sur 6.");Console.WriteLine($"Instance C avec rejets en chaîne ; résultat invariant à l'ordre des proposeurs={orderIndependent}.");

Appariage honnête (hommes proposent) : {m1: w2, m2: w3, m3: w1}


Hommes pouvant gagner par déclaration : 0 sur 3 ; C porte 3 bijections stables sur 6.


Instance C avec rejets en chaîne ; résultat invariant à l'ordre des proposeurs=True.


### (B) Rechercher une déclaration de répondante

Après les six déclarations testées pour chaque homme, énumérez les permutations des listes des femmes. Évaluez les partenaires selon leurs préférences vraies.

In [8]:
var part=C_honnete.ToDictionary(kv=>kv.Value,kv=>kv.Key);(string Woman,List<string> Declaration,string HonestPartner,int HonestRank,string NewPartner,int NewRank)? trouve=null;
foreach(var f in C_F.Keys){int baseRank=RangF(f,part[f]);foreach(var perm in Permute(C_F[f])){var fp2=new Dictionary<string,List<string>>(C_F);fp2[f]=perm;var m2=GaleShapley(C_H.Keys.ToList(),C_H,C_F.Keys.ToList(),fp2);var p2=m2.ToDictionary(kv=>kv.Value,kv=>kv.Key);if(RangF(f,p2[f])<baseRank){trouve=(f,perm,part[f],baseRank,p2[f],RangF(f,p2[f]));break;}}if(trouve is not null)break;}
if(trouve is {} t)Console.WriteLine($"Première répondante gagnante : {t.Woman}, honnête {t.HonestPartner} rang {t.HonestRank} → déclare [{string.Join(", ",t.Declaration)}] → {t.NewPartner} rang {t.NewRank}.");else Console.WriteLine("Aucune répondante gagnante dans cette instance.");
var manipulated=C_F.ToDictionary(kv=>kv.Key,kv=>kv.Value);manipulated["w2"]=new(){"m3","m2","m1"};var manipulatedMenMatch=GaleShapley(C_H.Keys.ToList(),C_H,C_F.Keys.ToList(),manipulated);var manipulatedWomenMatch=manipulatedMenMatch.ToDictionary(kv=>kv.Value,kv=>kv.Key);Console.WriteLine($"Rangs déclarés-vrais : w1 {RangF("w1",part["w1"])}→{RangF("w1",manipulatedWomenMatch["w1"])} ; w2 {RangF("w2",part["w2"])}→{RangF("w2",manipulatedWomenMatch["w2"])} ; w3 {RangF("w3",part["w3"])}→{RangF("w3",manipulatedWomenMatch["w3"])}.");
Console.WriteLine($"Rangs hommes : m1 {RangM("m1",C_honnete["m1"])}→{RangM("m1",manipulatedMenMatch["m1"])} ; m2 {RangM("m2",C_honnete["m2"])}→{RangM("m2",manipulatedMenMatch["m2"])} ; m3 {RangM("m3",C_honnete["m3"])}→{RangM("m3",manipulatedMenMatch["m3"])}.");

Première répondante gagnante : w2, honnête m1 rang 2 → déclare [m3, m2, m1] → m3 rang 1.


Rangs déclarés-vrais : w1 3→2 ; w2 2→1 ; w3 2→2.


Rangs hommes : m1 1→2 ; m2 2→2 ; m3 1→2.


### Résultat 3 — La non-stratégicité : qui peut tricher ?

La mesure **(A)** est surprenante : **aucun homme** ne peut améliorer son appariement en mentant sur ses préférences (`0 sur 3`, recherche exhaustive sur toutes les permutations de chaque liste). Le théorème d’optimalité du proposant porte sur les appariages stables du modèle classique. La recherche exhaustive ici ne porte que sur six déclarations par homme et cette instance 3×3. Le **côté proposeur est protégé**.

La mesure **(B)** révèle l'asymétrie : `w2`, honnête, obtient `m1` (son rang 2) ; en déclarant `[m3, m2, m1]` — elle ne fait que **repousser `m1` tout en bas** de sa liste — elle obtient `m3`, son rang 1. Le détail vaut une lecture : le partenaire gagné (`m3`) est le meilleur partenaire stable vrai de `w2` dans cette instance finie. Ce constat d’instance ne prouve pas une borne générale sur les manipulations des répondantes. Cette expérience finie montre une possibilité de gain pour une répondante dans C ; elle ne démontre pas une règle générale de manipulation pour le côté répondant.

Pourquoi cette asymétrie ? Côté proposeur, mentir réorganise la **propre** liste d'un homme, mais il reste dans la file de ceux qui **proposent** : l'algorithme lui attribue toujours un appariement stable, et son issue honnête est déjà son meilleur stable, si bien qu'aucun mensonge ne le fait monter au-dessus. Côté répondant, la situation diffère : une femme ne contrôle pas le moment où l'on lui propose, mais **changer l'ordre** de sa liste change qui lui est proposé en premier et qui est écarté. Elle peut ainsi se faire traiter par un sous-ensemble de prétendants plus favorable — son mensonge agit comme un **choix de qui court après elle**, ce que la règle honnête ne lui laissait pas. C'est exactement ce que fait la triche de `w2` : en repoussant `m1`, elle modifie la composition des prétendants qui la sollicitent, et remonte d'un rang. La protection du proposeur et l'exposition du répondant sont donc deux faces du même mécanisme de file de proposition.

L'algorithme n'est donc pas *sans stratagème* (*not strategy-proof*), mais **uniquement du côté des répondants**. D'où le principe de conception : **donner le pouvoir de proposer au côté qu'on veut protéger**. C'est exactement ce que fait le matching médical : les **résidents** (médecins en formation) sont le côté proposeur, précisément parce que l'algorithme les protège de leur propre manipulation stratégique, même s'il ne protège pas les hôpitaux.


## 4. Exercices

Les trois exercices ci-dessous font **reconstruire** l'algorithme et ses vérifications depuis zéro, pour que la non-stratégicité cesse d'être un résultat qu'on subit et devienne un mécanisme qu'on peut manipuler : implémenter la boucle de *deferred acceptance*, écrire un **vérificateur de stabilité** indépendant (qui ne suppose pas l'algorithme pour conclure), puis **chercher activement** un mensonge gagnant du côté répondant — l'expérience qui rend tangible l'asymétrie mesurée au résultat 3.


### Exercice 3 — Chercher une déclaration profitable

Le stub doit distinguer l’absence de gain sur A du gain mesuré pour w2 sur C. La garde vérifie que la liste retournée est complète et qu’elle produit réellement une amélioration.

In [9]:
#nullable enable
// Exercice 3 — stub de recherche de déclaration profitable.
List<string>? DeclarationProfitableEtudiante(string f,Dictionary<string,List<string>> proposerPrefs,Dictionary<string,List<string>> vraiRespondentPrefs)=>null; // TODO étudiant
bool HasProfitableLie(string woman,Dictionary<string,List<string>> pp,Dictionary<string,List<string>> truePrefs){var honest=GaleShapley(pp.Keys.ToList(),pp,truePrefs.Keys.ToList(),truePrefs);var honestReverse=honest.ToDictionary(kv=>kv.Value,kv=>kv.Key);int baseline=truePrefs[woman].IndexOf(honestReverse[woman]);foreach(var declaration in Permute(truePrefs[woman])){var report=new Dictionary<string,List<string>>(truePrefs);report[woman]=declaration;var changed=GaleShapley(pp.Keys.ToList(),pp,report.Keys.ToList(),report).ToDictionary(kv=>kv.Value,kv=>kv.Key);if(truePrefs[woman].IndexOf(changed[woman])<baseline)return true;}return false;}
var lieOnA=DeclarationProfitableEtudiante("w1",H_prefs,F_prefs);var lieOnC=DeclarationProfitableEtudiante("w2",C_H,C_F);bool liePresent=lieOnA is not null||lieOnC is not null;bool IsActualProfitableDeclaration(string woman,Dictionary<string,List<string>> proposerPrefs,Dictionary<string,List<string>> truePrefs,List<string>? declaration){if(declaration is null||declaration.Count!=truePrefs[woman].Count||declaration.Distinct().Count()!=declaration.Count||!declaration.OrderBy(x=>x).SequenceEqual(truePrefs[woman].OrderBy(x=>x)))return false;var honest=GaleShapley(proposerPrefs.Keys.ToList(),proposerPrefs,truePrefs.Keys.ToList(),truePrefs).ToDictionary(kv=>kv.Value,kv=>kv.Key);var reported=new Dictionary<string,List<string>>(truePrefs);reported[woman]=declaration;var changed=GaleShapley(proposerPrefs.Keys.ToList(),proposerPrefs,truePrefs.Keys.ToList(),reported).ToDictionary(kv=>kv.Value,kv=>kv.Key);return truePrefs[woman].IndexOf(changed[woman])<truePrefs[woman].IndexOf(honest[woman]);}
bool liePass=!HasProfitableLie("w1",H_prefs,F_prefs)&&lieOnA is null&&HasProfitableLie("w2",C_H,C_F)&&IsActualProfitableDeclaration("w2",C_H,C_F,lieOnC);
Console.WriteLine($"Exercice 3 — {(liePresent?(liePass?"réponse validée":"réponse incorrecte"):"à compléter")}; répondante sans gain sur A et gain sur C testés={liePass}; null est accepté sur A, mais pas sur w2 en C.");

Exercice 3 — à compléter; répondante sans gain sur A et gain sur C testés=False; null est accepté sur A, mais pas sur w2 en C.


## À retenir

Gale–Shapley termine sur des listes strictes complètes de même taille et produit un appariage stable. Dans A, l’énumération des six bijections trouve deux résultats stables ; dans C, elle en trouve trois. La proposition des hommes ou des femmes sélectionne des extrêmes différents, comme le prédit l’optimalité du côté proposant.

Dans C, aucune permutation déclarée par un homme ne l’améliore lors de l’énumération de cette instance. Une recherche exhaustive trouve pour w2 le mensonge `[m3, m2, m1]` : elle passe de m1 (rang 2) à m3 (rang 1). w1 s’améliore aussi (rang 3→2), w3 reste au rang 2 ; m1 et m3 perdent un rang et m2 est inchangé. La stabilité sous préférences déclarées n’établit pas la stabilité selon les préférences vraies. Ces observations finies illustrent le mécanisme ; les théorèmes généraux d’optimalité/protection du proposant restent des résultats distincts.

Le NRMP existe depuis 1952 ; la variante où les résidents proposent est utilisée depuis 1998. Les échanges rénaux utilisent des cycles et dépassent le modèle biparti simple. Gibbard–Satterthwaite porte sur les mécanismes de choix social et n’est pas directement le théorème de manipulation de ce modèle.

**Exercices à compléter** : [implémenter l’acceptation différée](#exercice-1), [vérifier une paire bloquante](#exercice-2), [chercher une déclaration gagnante](#exercice-3). Les réponses restent des stubs. Dans le jumeau C#, des fixtures contrôlent leurs résultats et distinguent un stub d’un `null` légitime.


## Vérifiez votre compréhension

1. L’instance A porte deux appariages stables. L’algorithme de Gale–Shapley est-il non déterministe ?

   <details><summary>Réponse</summary>

   Non. Avec un côté proposant et un ordre de traitement fixés, la procédure est déterministe ; elle sélectionne l’extrême stable optimal pour ce côté.

   </details>

2. L’énumération de C ne trouve aucun homme qui gagne en mentant. Est-ce le théorème général ?

   <details><summary>Réponse</summary>

   Non. La sortie est une vérification finie sur une seule instance et les permutations testées. L’optimalité du proposant est un théorème distinct du modèle d’appariage stable.

   </details>

3. Le mensonge de w2 donne-t-il aussi un gain aux autres femmes ?

   <details><summary>Réponse</summary>

   Dans cette instance, oui : w1 passe de rang 3 à 2. w3 reste au rang 2 ; m1 et m3 perdent un rang et m2 ne change pas. Ce relevé n’implique pas une loi générale de redistribution.

   </details>

4. L’issue est stable selon les listes déclarées. Est-elle nécessairement stable selon les préférences vraies ?

   <details><summary>Réponse</summary>

   Non. Un certificat de stabilité utilise les listes déclarées à l’algorithme. La manipulation peut modifier ces listes et la mesure des rangs vrais est une question distincte.

   </details>

## Pour aller plus loin

- **Gale & Shapley (1962)**, [*College Admissions and the Stability of Marriage*](https://doi.org/10.2307/2312726), référence fondatrice de l’acceptation différée.
- **Dubins & Freedman (1981)**, [*Machiavelli and the Gale-Shapley algorithm*](https://doi.org/10.1016/0304-4068(81)90021-3), résultat stratégique côté proposant.
- **NRMP**, [*How the Matching Algorithm Works*](https://www.nrmp.org/intro-to-the-match/how-matching-algorithm-works/) ; variante résidents-proposants utilisée depuis 1998, institution créée en 1952.
- [Version C#](GameTheory-16i-Appariage-Stable-Gale-Shapley-CSharp.ipynb), [carnet 16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb), [GameTheory-16h](GameTheory-16h-Cooperation-LLM-Python.ipynb), [GameTheory-16j](GameTheory-16j-Screening-Loyer-Information-Python.ipynb).
